#ENTRENAMIENTO DEL MODELO

En este notebook se entrena un modelo de clasificación para predecir si un vuelo se retrasará más de 15 minutos utilizando únicamente variables disponibles antes del despegue.
Se construye un baseline, se entrena un modelo, se evalúa su desempeño y se exporta el modelo final para producción.

##INTEGRA EL REPOSITORIO

In [1]:
%cd /content
!rm -rf FlightOnTime-

/content


In [2]:
!git clone https://github.com/DnRiv/FlightOnTime-.git
%cd FlightOnTime-/ds

Cloning into 'FlightOnTime-'...
remote: Enumerating objects: 3031, done.
remote: Counting objects: 100% (89/89), done.
remote: Compressing objects: 100% (67/67), done.
remote: Total 3031 (delta 13), reused 64 (delta 10), pack-reused 2942 (from 2)
Receiving objects: 100% (3031/3031), 18.73 MiB | 5.08 MiB/s, done.
Resolving deltas: 100% (1128/1128), done.
Updating files: 100% (88/88), done.
/content/FlightOnTime-/ds


In [3]:
!pip install -r requirements.txt

In [4]:
import pandas as pd
import os
import joblib

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

from src.preprocessing import build_dataset
import numpy as np
from sklearn.metrics import accuracy_score

In [5]:
import sklearn
sklearn.__version__


'1.3.0'

##CARGA DEL DATASET

Cargamos el dataset limpio y con feature engineering aplicado desde preprocessing.py.

In [6]:
!pip install kagglehub[pandas-datasets]

In [7]:
import os
# Requiere configurar la variable de entorno KAGGLE_API_TOKEN
os.environ["KAGGLE_API_TOKEN"] = ""

In [8]:
import kagglehub
from kagglehub import KaggleDatasetAdapter

df = kagglehub.dataset_load(
    KaggleDatasetAdapter.PANDAS,
    "heemalichaudhari/airlines-delay",
    "DelayedFlights.csv"
)

df.head()

Using Colab cache for faster access to the 'airlines-delay' dataset.


,Unnamed: 0,Year,Month,DayofMonth,DayOfWeek,DepTime,CRSDepTime,ArrTime,CRSArrTime,UniqueCarrier,...,TaxiIn,TaxiOut,Cancelled,CancellationCode,Diverted,CarrierDelay,WeatherDelay,NASDelay,SecurityDelay,LateAircraftDelay
0,0,2008,1,3,4,2003.0,1955,2211.0,2225,WN,...,4.0,8.0,0,N,0,NaN,NaN,NaN,NaN,NaN
1,1,2008,1,3,4,754.0,735,1002.0,1000,WN,...,5.0,10.0,0,N,0,NaN,NaN,NaN,NaN,NaN
2,2,2008,1,3,4,628.0,620,804.0,750,WN,...,3.0,17.0,0,N,0,NaN,NaN,NaN,NaN,NaN
3,4,2008,1,3,4,1829.0,1755,1959.0,1925,WN,...,3.0,10.0,0,N,0,2.0,0.0,0.0,0.0,32.0
4,5,2008,1,3,4,1940.0,1915,2121.0,2110,WN,...,4.0,10.0,0,N,0,NaN,NaN,NaN,NaN,NaN


##NORMALIZACIÓN DE NOMBRES

Unificamos nombres de columnas para mantener consistencia entre entrenamiento y API.

In [9]:
df.rename(columns={
    "UniqueCarrier": "Unique_carrier",
    "Origin": "Origin",
    "Dest": "Destination",
    "DayOfWeek": "Day_of_week",
    "dep_hour": "Dep_hour",
    "Distance": "Distance_miles"
}, inplace=True)


In [10]:
print(df.columns)


Index(['Unnamed: 0', 'Year', 'Month', 'DayofMonth', 'Day_of_week', 'DepTime',
       'CRSDepTime', 'ArrTime', 'CRSArrTime', 'Unique_carrier', 'FlightNum',
       'TailNum', 'ActualElapsedTime', 'CRSElapsedTime', 'AirTime', 'ArrDelay',
       'DepDelay', 'Origin', 'Destination', 'Distance_miles', 'TaxiIn',
       'TaxiOut', 'Cancelled', 'CancellationCode', 'Diverted', 'CarrierDelay',
       'WeatherDelay', 'NASDelay', 'SecurityDelay', 'LateAircraftDelay'],
      dtype='object')


##CONSTRUCCIÓN DE X, y PRERPOCESSOR

Construimos el dataset final de entrenamiento usando el pipeline de preprocessing.

In [11]:
X, y, preprocessor = build_dataset(df)

In [12]:
print(X.shape)
print(y.shape)

X.isna().sum().sum()
y.isna().sum()


(1928371, 6)
(1928371,)


0

##BASELINE

El baseline permite comparar si el modelo realmente aprende algo útil.

In [13]:
#Baseline: siempre predecir "no retrasado"
baseline_pred = np.zeros(len(y))
print("Baseline accuracy:", accuracy_score(y, baseline_pred))


Baseline accuracy: 0.369928296992643


##TRAIN / TEST SPLIT

Separación estratificada para mantener proporción de vuelos retrasados.

In [14]:
#Train,Test
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

##PIPELINE DE ENTRENAMIENTO

Pipeline completo: transformación + modelo en un solo objeto reproducible.

In [15]:
#Pipeline
model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", LogisticRegression(max_iter=1000))
    ]
)

##ENTRENAMIENTO

In [16]:
model.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['Unique_carrier', 'Origin',
                                                   'Destination',
                                                   'Day_of_week']),
                                                 ('num', StandardScaler(),
                                                  ['Dep_hour',
                                                   'Distance_miles'])])),
                ('classifier', LogisticRegression(max_iter=1000))])

##EVALUACIÓN

Se evalúa precisión, recall y AUC para medir capacidad predictiva real.

In [17]:
y_pred = model.predict(X_test)

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.55      0.14      0.23    142672
           1       0.65      0.93      0.77    243003

    accuracy                           0.64    385675
   macro avg       0.60      0.54      0.50    385675
weighted avg       0.61      0.64      0.57    385675



In [18]:
from sklearn.metrics import roc_auc_score

y_prob = model.predict_proba(X_test)[:, 1]
print("ROC AUC:", roc_auc_score(y_test, y_prob))


ROC AUC: 0.6112741463479667


##AJUSTE DE HYPERPARÁMETROS

Ajustamos hiperparámetros para mejorar recall de vuelos retrasados.

In [19]:
#Ajuste modelo balanceado
model_tuned = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", LogisticRegression(max_iter=2000, C=0.5, class_weight="balanced"))
    ]
)

model_tuned.fit(X_train, y_train)

y_pred_tuned = model_tuned.predict(X_test)
print(classification_report(y_test, y_pred_tuned))


              precision    recall  f1-score   support

           0       0.45      0.57      0.50    142672
           1       0.70      0.58      0.64    243003

    accuracy                           0.58    385675
   macro avg       0.57      0.58      0.57    385675
weighted avg       0.61      0.58      0.59    385675



El modelo prioriza detectar vuelos retrasados (recall alto) para maximizar capacidad de alerta temprana, aceptando un mayor número de falsos positivos.



In [21]:
#Export
import os
import joblib

artifacts_path = "artifacts"
os.makedirs(artifacts_path, exist_ok=True)

model_path = os.path.join(artifacts_path, "model.joblib")
joblib.dump(model, model_path)

print("Modelo exportado en:", model_path)


Modelo exportado en: artifacts/model.joblib


##CONCLUSIÓN

Se entrenó un modelo de clasificación para predecir retrasos de vuelos utilizando únicamente información disponible antes del despegue.
El modelo supera ampliamente el baseline y muestra una capacidad predictiva consistente, por lo que se exporta para su uso en producción a través de la API.